# Chapter 2 notebook — PyTorch → ONNX export

Walks through the Chapter 5 doc (`docs/05_pytorch_to_onnx.md`) interactively:
1. Export a torchvision classifier to ONNX (static, then dynamic).
2. Validate the file with `onnx.checker`.
3. Compare PyTorch and ONNX Runtime outputs (max abs diff, cosine similarity).
4. Look at the ONNX graph briefly.

Run this from the repo root (so the relative paths resolve).

In [ ]:
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import torch
from torchvision import models

print('torch', torch.__version__, '| onnx', onnx.__version__, '| onnxruntime', ort.__version__)
print('available providers:', ort.get_available_providers())

OUT_DIR = Path('../experiments/exported_models')
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load a model in eval mode

In [ ]:
weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
model = models.mobilenet_v3_small(weights=weights).eval()
dummy = torch.randn(1, 3, 224, 224)
print(f'params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M')

## 2. Static-shape export

In [ ]:
static_path = OUT_DIR / 'mobilenet_v3_small.onnx'
torch.onnx.export(
    model, dummy, static_path.as_posix(),
    input_names=['input'], output_names=['logits'],
    opset_version=17, do_constant_folding=True,
    dynamo=False,                # keep everything in one file
)
size_mb = static_path.stat().st_size / 1024 / 1024
print(f'wrote {static_path}  ({size_mb:.2f} MB)')

## 3. Validate with `onnx.checker`

In [ ]:
m = onnx.load(static_path.as_posix())
onnx.checker.check_model(m)
print('ir_version:', m.ir_version)
print('opset:', m.opset_import[0].version)
print('input:', m.graph.input[0].name, [d.dim_value for d in m.graph.input[0].type.tensor_type.shape.dim])
print('output:', m.graph.output[0].name, [d.dim_value for d in m.graph.output[0].type.tensor_type.shape.dim])

## 4. Numerical equivalence: PyTorch vs ONNX Runtime

In [ ]:
x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    y_torch = model(x).numpy()

sess = ort.InferenceSession(static_path.as_posix(), providers=['CPUExecutionProvider'])
y_ort = sess.run(None, {'input': x.numpy()})[0]

max_diff = float(np.abs(y_torch - y_ort).max())
cosine = float((y_torch.flatten() @ y_ort.flatten()) / (np.linalg.norm(y_torch) * np.linalg.norm(y_ort) + 1e-12))
same_argmax = int(np.argmax(y_torch, 1)[0]) == int(np.argmax(y_ort, 1)[0])
print(f'max|diff|={max_diff:.6f}   cosine={cosine:.6f}   same_argmax={same_argmax}')
assert max_diff < 1e-3 and cosine > 0.999

## 5. Dynamic-shape export

Now allow variable batch and spatial dims. Useful when you want one ONNX file to serve multiple input sizes.

In [ ]:
dynamic_path = OUT_DIR / 'mobilenet_v3_small-dyn.onnx'
torch.onnx.export(
    model, dummy, dynamic_path.as_posix(),
    input_names=['input'], output_names=['logits'],
    dynamic_axes={
        'input':  {0: 'batch', 2: 'height', 3: 'width'},
        'logits': {0: 'batch'},
    },
    opset_version=17, do_constant_folding=True, dynamo=False,
)
print(f'wrote {dynamic_path}  ({dynamic_path.stat().st_size / 1024 / 1024:.2f} MB)')

sess_dyn = ort.InferenceSession(dynamic_path.as_posix(), providers=['CPUExecutionProvider'])
for shape in [(1, 3, 224, 224), (4, 3, 224, 224), (1, 3, 192, 192), (2, 3, 256, 256)]:
    xd = np.random.randn(*shape).astype(np.float32)
    y = sess_dyn.run(None, {'input': xd})[0]
    print(f'  input={shape}  output={y.shape}')

## 6. Brief look at the ONNX graph

In [ ]:
graph_text = onnx.helper.printable_graph(m.graph)
print(graph_text[:1500])
print('...')
print(f'(graph has {len(m.graph.node)} ops)')

## Take-aways

1. `torch.onnx.export(..., dynamo=False)` is the simple, single-file legacy path used in this course.
2. **Always validate** with `onnx.checker` and a numerical diff against PyTorch.
3. Use `dynamic_axes` if you need variable batch / spatial sizes; expect a small perf cost on TensorRT.
4. Open the `.onnx` file in [Netron](https://netron.app) for visual inspection of any layer fusions.

Next: **Chapter 3 notebook** loads this ONNX file via ONNX Runtime, swaps execution providers, and benchmarks them.